# Phase 6 — vérification

Le contrôle reconstruit le suivi, puis vérifie les totaux, les trois expositions demandées, et l'accord avec les métriques de la phase 5. Il ne fixe ni le nombre de titres, ni le mois du rapport.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.portfolio_construction.build_portfolios import (
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    STRATEGY_IDS,
    WEIGHTS_FILE,
)
from src.portfolio_monitoring.build_monitoring import (
    POSITION_WEIGHT_FLOOR,
    REQUESTED_EXPOSURES,
    TOP_POSITIONS,
    WEIGHT_SUM_TOLERANCE,
    build_monitoring,
)

written = build_monitoring()
report = pd.read_csv(written["report"])
sectors = pd.read_csv(written["sectors"])
zones = pd.read_csv(written["zones"])
exposure = pd.read_csv(written["exposure"])
concentration = pd.read_csv(written["concentration"])
top10 = pd.read_csv(written["top10"])
holdings = pd.read_csv(written["holdings"], dtype={"ticker": str, "yahoo_symbol": str})
weights = pd.read_csv(WEIGHTS_FILE, dtype={"ticker": str, "yahoo_symbol": str})
metrics = pd.read_csv(METRICS_FILE)

assert set(report["strategy"]) == set(STRATEGY_IDS)
assert report["report_month"].nunique() == 1
assert report["report_month"].eq(report["window_end"]).all()
assert holdings["weight"].ge(0).all()
assert not holdings["sector"].eq("").any()
assert not holdings["zone"].eq("").any()
assert not holdings["exposure"].eq("").any()

for strategy in STRATEGY_IDS:
    total = float(weights.loc[weights["strategy"].eq(strategy), "weight"].sum())
    assert abs(total - 1) <= WEIGHT_SUM_TOLERANCE
    for table in (sectors, zones, exposure):
        grouped = float(table.loc[table["strategy"].eq(strategy), "weight"].sum())
        assert abs(grouped - 1) <= 1e-6
    labels = set(exposure.loc[exposure["strategy"].eq(strategy), "exposure"])
    assert set(REQUESTED_EXPOSURES).issubset(labels)
    published = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    monitored = concentration.loc[concentration["strategy"].eq(strategy)].iloc[0]
    assert int(monitored["names"]) == int(published["names"])
    assert abs(monitored["maximum_weight"] - published["maximum_weight"]) <= 1e-6
    assert abs(monitored["effective_holdings"] - published["effective_holdings"]) <= 1e-6
    assert abs(monitored["hhi"] * monitored["effective_holdings"] - 1) <= 1e-6
    assert abs(monitored["concentration_multiple"] - monitored["hhi"] * monitored["names"]) <= 1e-6
    assert monitored["invested_names"] <= monitored["names"]
    ranks = top10.loc[top10["strategy"].eq(strategy), "rank"].astype(int).tolist()
    assert ranks == list(range(1, len(ranks) + 1))
    assert 1 <= len(ranks) <= TOP_POSITIONS
    assert top10.loc[top10["strategy"].eq(strategy), "weight"].gt(POSITION_WEIGHT_FLOOR).all()

equal = concentration.loc[concentration["strategy"].eq("equal_weight")].iloc[0]
assert abs(equal["concentration_multiple"] - 1) <= 1e-6
assert int(equal["invested_names"]) == int(equal["names"])

path = pd.read_csv(written["path"])
contributions = pd.read_csv(written["contributions"])
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
for strategy in STRATEGY_IDS:
    series = path.loc[path["strategy"].eq(strategy)].sort_values("month")
    official = monthly.loc[monthly["strategy"].eq(strategy)].sort_values("month")
    assert list(series["month"]) == list(official["month"])
    assert abs(series["portfolio_return"] - official["portfolio_return"]).max() <= 1e-8
    wealth = float((1 + series["portfolio_return"]).prod())
    assert abs(series["cumulative_wealth"].iloc[-1] - wealth) <= 1e-8
    assert series["drawdown"].max() <= 1e-12
    grouped = contributions.loc[contributions["strategy"].eq(strategy)]
    summed = grouped.groupby("month")["month_contribution"].sum()
    compared = series.set_index("month")["portfolio_return"]
    assert abs(summed.reindex(compared.index) - compared).max() <= 1e-6

from src.portfolio_monitoring.dashboard import DASHBOARD_FILE, SECTION_TITLES

dashboard_html = DASHBOARD_FILE.read_text(encoding="utf-8")
for title in SECTION_TITLES:
    assert title.split()[0] in dashboard_html
assert str(report["report_month"].iloc[0]) in dashboard_html

print(
    f"Rapport : {report['report_month'].iloc[0]}. "
    f"Stratégies : {report['strategy'].nunique()}."
)
print("Résultat : PASS")


Rapport : 2026-09. Stratégies : 3.
Résultat : PASS
